In [ ]:
#%pip install soccerdata
%pip install webdriver_manager

  Using cached setuptools-82.0.1-py3-none-any.whl.metadata (6.5 kB)
   ---------------------------------------- 0.0/663.6 kB ? eta -:--:--
   ------------------------------- -------- 524.3/663.6 kB 6.9 MB/s eta 0:00:01
   ---------------------------------------- 663.6/663.6 kB 2.8 MB/s  0:00:00
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/9.6 MB 5.1 MB/s eta 0:00:02
   ------- -------------------------------- 1.8/9.6 MB 4.5 MB/s eta 0:00:02
   ------------ --------------------------- 2.9/9.6 MB 4.7 MB/s eta 0:00:02
   --------------- ------------------------ 3.7/9.6 MB 4.7 MB/s eta 0:00:02
   -------------------- ------------------- 5.0/9.6 MB 4.8 MB/s eta 0:00:01
   ----------------------- ---------------- 5.5/9.6 MB 4.6 MB/s eta 0:00:01
   --------------------------- ------------ 6.6/9.6 MB 4.5 MB/s eta 0:00:01
   ----------------------------- ---------- 7.1/9.6 MB 4.4 MB/s eta 0:00:01
   ------------------------

In [ ]:
import pandas as pd
from io import StringIO
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.library import ChromeDriverManager
import time

def get_fbref_table(url):
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")  # pas de fenêtre visible
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    
    driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)
    driver.get(url)
    time.sleep(3)  # attendre le chargement
    html = driver.page_source
    driver.quit()
    return html

stat_urls = {
    "standard":   "https://fbref.com/en/comps/13/stats/Ligue-1-Stats",
    "shooting":   "https://fbref.com/en/comps/13/shooting/Ligue-1-Stats",
    "passing":    "https://fbref.com/en/comps/13/passing/Ligue-1-Stats",
    "defense":    "https://fbref.com/en/comps/13/defense/Ligue-1-Stats",
    "possession": "https://fbref.com/en/comps/13/possession/Ligue-1-Stats",
    "gca":        "https://fbref.com/en/comps/13/gca/Ligue-1-Stats",
}

dfs = {}
for stat_name, url in stat_urls.items():
    print(f"Récupération : {stat_name}...")
    try:
        html = get_fbref_table(url)
        tables = pd.read_html(StringIO(html))
        
        for t in tables:
            cols = [str(c).lower() for c in t.columns.get_level_values(-1)]
            if "player" in cols:
                df = t
                break
        
        # Aplatir les headers multi-niveaux
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = ["_".join(filter(lambda x: "Unnamed" not in str(x), col)).strip("_")
                          or col[-1] for col in df.columns]
        
        df = df[df["Player"] != "Player"].dropna(subset=["Player"]).reset_index(drop=True)
        dfs[stat_name] = df
        print(f"  ✅ {len(df)} joueurs")
        time.sleep(4)  # rate limit
    
    except Exception as e:
        print(f"  ❌ {e}")

# Fusion
if dfs:
    base = dfs["standard"]
    for stat_name, df in dfs.items():
        if stat_name == "standard":
            continue
        cols_to_add = [c for c in df.columns if c not in base.columns or c in ["Player", "Squad"]]
        base = base.merge(df[cols_to_add], on=["Player", "Squad"], how="left")
    
    base.to_csv("ligue1_players_all_stats.csv", index=False)
    print(f"\n✅ Sauvegardé : {len(base)} joueurs × {len(base.columns)} colonnes")

[04/15/26 07:51:38] INFO     No custom team name replacements found. You can configure these in       _config.py:92
                             C:\Users\LouisHarle\soccerdata\config\teamname_replacements.json.                     

                    INFO     No custom league dict found. You can configure additional leagues in    _config.py:190
                             C:\Users\LouisHarle\soccerdata\config\league_dict.json.                               

                    INFO     Saving cached data to C:\Users\LouisHarle\soccerdata\data\FBref         _common.py:250



*** chromedriver to download = 147.0.7727.56 (Latest Stable) 

https://storage.googleapis.com/chrome-for-testing-public/147.0.7727.56/win64/chromedriver-win64.zip ...
Download Complete!

Extracting ['chromedriver.exe'] from chromedriver-win64.zip ...
Unzip Complete!

The file [uc_driver.exe] was saved to:
C:\Users\LouisHarle\OneDrive - Quadratic\Bureau\Stage_VM\.venv\Lib\site-packages\seleniumbase\drivers\
uc_driver.exe

Making [uc_driver.exe 147.0.7727.56] executable ...
[uc_driver.exe 147.0.7727.56] is now ready for use!



Exception: Chrome not found! Install it first!